In [20]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings


from pathlib import Path

# 현재 작업폴더 기준으로 상위 폴더들 중 'src'가 있는 경로를 찾아 추가
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "src").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        print("프로젝트 루트 추가됨:", p)
        break
else:
    raise FileNotFoundError("상위 폴더들에서 'src' 폴더를 찾지 못했습니다.")

from src.utils.model_manager import ModelManager

프로젝트 루트 추가됨: c:\skn24\SKN24-2nd-3Team


In [21]:
df = pd.read_csv('../../data/processed/ljh_preprocessed.csv')
df

,user_id,country_id,gender,age_group,reg_date,first_deposit,first_bet,fixed_bet_amount,live_bet_amount,total_bet_amount,...,fixed_hit_days,live_hit_days,total_hit_days,fixed_win_rate,live_win_rate,total_win_rate,fixed_avg_roi,live_avg_roi,total_avg_roi,churn
0,1324354,276.0,1.0,3.0,2005-02-01,2005-02-24,2005-02-24,15750.38,2146.47,17896.85,...,93.0,22.0,29.0,0.24,0.42,0.27,-0.00,-0.09,-0.08,0
1,1324355,300.0,1.0,1.0,2005-02-01,2005-02-01,2005-02-01,639.30,24.70,664.00,...,15.0,1.0,1.0,0.14,0.14,0.20,0.17,-0.76,-0.77,0
2,1324356,276.0,1.0,2.0,2005-02-01,2005-02-01,2005-02-02,898.81,701.82,1600.63,...,12.0,18.0,18.0,0.12,0.26,0.33,0.38,-0.25,0.64,0
3,1324358,752.0,1.0,1.0,2005-02-01,2005-02-01,2005-02-01,247.70,88.59,336.29,...,1.0,1.0,1.0,0.00,0.00,0.00,-0.88,-0.37,-0.54,1
4,1324360,792.0,1.0,2.0,2005-02-01,2005-02-02,2005-02-02,685.94,6.66,692.61,...,14.0,2.0,3.0,0.14,0.25,0.50,0.89,-0.63,6.99,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46334,1405183,276.0,1.0,3.0,2005-02-28,2005-03-03,2005-03-04,88.76,102.13,190.89,...,1.0,8.0,8.0,0.04,0.23,0.17,-0.90,-0.29,-0.44,1
46335,1405184,276.0,1.0,2.0,2005-02-28,2005-02-28,2005-03-01,315.88,296.75,612.63,...,11.0,15.0,17.0,0.24,0.39,0.43,-0.46,-0.21,-0.09,0
46336,1405185,276.0,1.0,2.0,2005-02-28,2005-11-08,2005-11-08,13.00,124.71,137.71,...,1.0,2.0,1.0,0.50,0.20,0.00,-0.20,-0.59,-0.73,0
46337,1405189,276.0,1.0,1.0,2005-02-28,2005-03-01,2005-03-01,414.09,10.00,424.09,...,32.0,2.0,2.0,0.28,0.67,1.00,0.20,0.47,1.35,0


In [22]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 46339 entries, 0 to 46338
Data columns (total 29 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   user_id            46339 non-null  int64  
 1   country_id         46337 non-null  float64
 2   gender             46337 non-null  float64
 3   age_group          46337 non-null  float64
 4   reg_date           46337 non-null  str    
 5   first_deposit      46337 non-null  str    
 6   first_bet          46337 non-null  str    
 7   fixed_bet_amount   46337 non-null  float64
 8   live_bet_amount    46337 non-null  float64
 9   total_bet_amount   46337 non-null  float64
 10  fixed_win_amount   46337 non-null  float64
 11  live_win_amount    46337 non-null  float64
 12  total_win_amount   46337 non-null  float64
 13  fixed_bet_cnt      46337 non-null  float64
 14  live_bet_cnt       46337 non-null  float64
 15  total_bet_cnt      46337 non-null  float64
 16  fixed_active_days  46337 non-null

In [23]:
# X, y 분리
X = df.drop(columns=['user_id', 'churn', 'country_id'])
y = df['churn']

# 날씨 컬럼(문자열) 제거
X = X.drop(columns=['reg_date', 'first_deposit', 'first_bet'])

In [24]:
# X, y 데이터 분리
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)   # 불균형 데이터라 stratify=y 추가

# 결측치 처리
train_median = X_train.median(numeric_only=True)
X_train = X_train.fillna(train_median)
X_test = X_test.fillna(train_median)

In [25]:
# KNN
from sklearn.neighbors import KNeighborsClassifier
kn = KNeighborsClassifier()
kn.fit(X_train, y_train)

print('모델명: KNN')
print('학습 데이터 점수:', kn.score(X_train, y_train))
print('평가 데이터 점수:', kn.score(X_test, y_test))

모델명: KNN
학습 데이터 점수: 0.8365299020797928
평가 데이터 점수: 0.7706085455330168


In [26]:
# KNN 스케일링
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_scaler = scaler.fit_transform(X_train)
X_test_scaler = scaler.transform(X_test)
kn.fit(X_train_scaler, y_train)

print('모델명: KNN_Scaler')
print('학습 데이터 점수:', kn.score(X_train_scaler, y_train))
print('평가 데이터 점수:', kn.score(X_test_scaler, y_test))

모델명: KNN_Scaler
학습 데이터 점수: 0.8406031668959564
평가 데이터 점수: 0.7777298230470436


In [27]:
# 하이퍼 파라미터 튜닝
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
kn = KNeighborsClassifier()

params = {'n_neighbors': range(1, 15, 2)}
grid = GridSearchCV(kn, params, scoring='recall', cv=5)
grid.fit(X_train_scaler, y_train)
y_pred = grid.best_estimator_.predict(X_test_scaler)

print('최적의 하이퍼 파라미터:', grid.best_params_)
print('최고 CV 점수(recall):', grid.best_score_)
print('학습 데이터 점수:', grid.best_estimator_.score(X_train_scaler,y_train))
print('평가 데이터 점수:', grid.best_estimator_.score(X_test_scaler,y_test))
print(classification_report(y_test, y_pred))

최적의 하이퍼 파라미터: {'n_neighbors': 5}
최고 CV 점수(recall): 0.5313031756550789
학습 데이터 점수: 0.8406031668959564
평가 데이터 점수: 0.7777298230470436
              precision    recall  f1-score   support

           0       0.83      0.87      0.85      6665
           1       0.62      0.54      0.58      2603

    accuracy                           0.78      9268
   macro avg       0.72      0.70      0.71      9268
weighted avg       0.77      0.78      0.77      9268



In [28]:
# 하이퍼 파라미터 상위 5개 출력
cv_df = pd.DataFrame(grid.cv_results_)

# 필요한 컬럼만 선택
top5 = cv_df[['params', 'mean_test_score', 'rank_test_score']].copy()

# 순위 정렬
top5 = top5.sort_values('rank_test_score').head(5)

# 컬럼명 변경
top5 = top5.rename(columns={
    'params': '주요 파라미터',
    'mean_test_score': 'CV Score (Recall)',
    'rank_test_score': '순위'
})

print(top5)

              주요 파라미터  CV Score (Recall)  순위
2  {'n_neighbors': 5}           0.531303   1
1  {'n_neighbors': 3}           0.526694   2
0  {'n_neighbors': 1}           0.523719   3
3  {'n_neighbors': 7}           0.521317   4
4  {'n_neighbors': 9}           0.516132   5


In [29]:
# confusion_matrix
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
best_kn = grid.best_estimator_

y_pred = best_kn.predict(X_test_scaler)
y_proba = best_kn.predict_proba(X_test_scaler)[:, 1]    # roc_auc용 확률값

# confusion matrix
print('[Confusion Matrix]')
print(confusion_matrix(y_test, y_pred))

# classification report
print('[Classification Report]')
print(classification_report(y_test, y_pred))

# 주요 지표
print('[평가 지표]')
print('평가 데이터 정확도:', accuracy_score(y_test, y_pred))
print('평가 데이터 정밀도:', precision_score(y_test, y_pred))
print('평가 데이터 재현율:', recall_score(y_test, y_pred))
print('평가 데이터 F1-score:', f1_score(y_test, y_pred))
print('평가 데이터 ROC-AUC:', roc_auc_score(y_test, y_proba))

[Confusion Matrix]
[[5813  852]
 [1208 1395]]
[Classification Report]
              precision    recall  f1-score   support

           0       0.83      0.87      0.85      6665
           1       0.62      0.54      0.58      2603

    accuracy                           0.78      9268
   macro avg       0.72      0.70      0.71      9268
weighted avg       0.77      0.78      0.77      9268

[평가 지표]
평가 데이터 정확도: 0.7777298230470436
평가 데이터 정밀도: 0.6208277703604806
평가 데이터 재현율: 0.5359200922013062
평가 데이터 F1-score: 0.5752577319587628
평가 데이터 ROC-AUC: 0.7993193842064051


In [30]:
# baseline
baseline_kn = KNeighborsClassifier()
baseline_kn.fit(X_train_scaler, y_train)

y_pred_base = baseline_kn.predict(X_test_scaler)
y_proba_base = baseline_kn.predict_proba(X_test_scaler)[:, 1]

# baseline confusion matrix
print('[Baseline Confusion Matrix]')
print(confusion_matrix(y_test, y_pred_base))

# baseline classification report
print('[Baseline Classification Report]')
print(classification_report(y_test, y_pred_base))

# 주요 지표
print('[Baseline 평가 지표]')
print('Accuracy :', accuracy_score(y_test, y_pred_base))
print('Precision:', precision_score(y_test, y_pred_base))
print('Recall   :', recall_score(y_test, y_pred_base))
print('F1-score :', f1_score(y_test, y_pred_base))
print('ROC-AUC  :', roc_auc_score(y_test, y_proba_base))

[Baseline Confusion Matrix]
[[5813  852]
 [1208 1395]]
[Baseline Classification Report]
              precision    recall  f1-score   support

           0       0.83      0.87      0.85      6665
           1       0.62      0.54      0.58      2603

    accuracy                           0.78      9268
   macro avg       0.72      0.70      0.71      9268
weighted avg       0.77      0.78      0.77      9268

[Baseline 평가 지표]
Accuracy : 0.7777298230470436
Precision: 0.6208277703604806
Recall   : 0.5359200922013062
F1-score : 0.5752577319587628
ROC-AUC  : 0.7993193842064051


In [31]:
# train, test f1_Score 찾기
# 최적모델
y_pred_train = best_kn.predict(X_train_scaler)
train_f1 = f1_score(y_train, y_pred_train)

print("Train F1-score:", train_f1)
print("Test F1-score :", f1_score(y_test, y_pred))

Train F1-score: 0.694861864187968
Test F1-score : 0.5752577319587628


In [32]:
from src.utils.model_manager import ModelManager
mm = ModelManager(base_dir='../../models/knn_churn_v1')

# 최종 선택 모델의 성능 지표 정리 (임계값 최적화 적용 기준)
final_metrics = {
    'accuracy':  accuracy_score(y_test, y_pred_base),
    'precision': precision_score(y_test, y_pred_base),
    'recall':    recall_score(y_test, y_pred_base),
    'f1_score':  f1_score(y_test, y_pred_base),
    'roc_auc':   roc_auc_score(y_test, y_proba_base),
}

mm.save(
    grid.best_estimator_,
    'knn',
    metadata={
        **final_metrics,
        'best_params':      grid.best_params_,       # GridSearchCV 최적 파라미터
        'best_cv_score':      grid.best_score_,        # CV 최고 점수 (현재 scoring='recall')
        'features':         X.columns.tolist(),      # 학습에 사용된 피처 목록
        'n_features':       X.shape[1],
        'train_size':       len(X_train),
        'test_size':        len(X_test)
    }
)

print('\n=== 저장된 모델 목록 ===')
for m in mm.list_models():
    name   = m.get('model_name', 'N/A')
    custom = m.get('custom', {})
    f1     = custom.get('f1_score', 0)
    auc    = custom.get('roc_auc', 0)
    print(f'  [{name}]  F1={f1:.4f}  |  ROC-AUC={auc:.4f}')

[Model Manager]: knn saved successfully (3.1 MB)

=== 저장된 모델 목록 ===
  [knn]  F1=0.5753  |  ROC-AUC=0.7993
